In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Wan 终端投影写读迁移：首轮真实验证

复用已保存共享终端，在第一次 Wan VAE 解码前写入。固定 margin=1、4组×16通道×4×4，1760个support，两条固定消息模板。

请选择 GPU 运行时，按顺序运行所有单元。无需生成新内容：0 Transformer、3 VAE decode、28 VAE encode。正常、匹配二次保存、固定删除138，共七条接收视频。结果与失败证据保存到 Drive。

此 notebook 已做 schema/语法与 CPU 数组检查，尚未运行真实模型。正常失败先查投影承载；删除与重存对照比较。没有 PASS/FPR 阈值，也不宣称盲定位单帧删除。

## 拉取固定源码与安装

源码 SHA：`080635978f2c6b78633c0ed0fc8e83f157ed5599`。复用已跑通的依赖、加载与子进程路径。

In [ ]:
from pathlib import Path
import sys,subprocess
URL='https://github.com/RICHAAARC/SC-SSTW.git'; REF='080635978f2c6b78633c0ed0fc8e83f157ed5599'; SOURCE=Path('/content/wan_projection_source')
if SOURCE.exists():
    if subprocess.check_output(['git','-C',str(SOURCE),'remote','get-url','origin'],text=True).strip()!=URL: raise RuntimeError('unexpected origin')
else:
    subprocess.run(['git','init',str(SOURCE)],check=True); subprocess.run(['git','-C',str(SOURCE),'remote','add','origin',URL],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','--depth','1','origin',REF],check=True); subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','--force','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','diffusers','transformers','accelerate','ftfy','sentencepiece','safetensors','huggingface_hub','numpy','Pillow'],check=True); subprocess.run(['ffmpeg','-version'],check=True)
print('Pinned source:', subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip())


## 运行并持续落盘

输入终端：`/content/drive/MyDrive/Video-WM/C2T1/c2t1_20260915T092031Z/shared_terminal_normalized.pt`。保留源码 config 中的固定候选；默认直接执行。每个运行使用独立目录，日志写在目录同级。

In [ ]:
from datetime import datetime,timezone
CONFIG=SOURCE/'runtime/tstwv2/first_validation.json'; RUN_ID='wan_projection_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); OUTPUT=Path('/content/drive/MyDrive/Video-WM/WanProjection')/RUN_ID
if OUTPUT.exists(): raise FileExistsError(OUTPUT)
import os,signal
cmd=[sys.executable,'-u','-m','runtime.tstwv2.run','--config',str(CONFIG),'--output',str(OUTPUT)]; LOG=OUTPUT.parent/f'{RUN_ID}.launcher.log'; LOG.parent.mkdir(parents=True,exist_ok=True)
with LOG.open('w') as log:
    p=subprocess.Popen(cmd,cwd=SOURCE,start_new_session=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout: print(line,end=''); log.write(line); log.flush()
        code=p.wait()
    except BaseException:
        try: p.send_signal(signal.SIGTERM)
        except ProcessLookupError: pass
        try: p.wait(timeout=5)
        except subprocess.TimeoutExpired: os.killpg(p.pid,signal.SIGKILL); p.wait()
        raise
print('launcher exit', code)
print('Results:', OUTPUT)
print('Log:', LOG)
if code: raise subprocess.CalledProcessError(code,cmd)


## 查看结果

完整投影码本、写前/写后测量、MP4、重编码 latent 与候选表均位于输出目录。固定单帧删除只检验显式仿射读出的容忍能力，未引入局部路径或补帧。

In [ ]:
import json
result=json.loads((OUTPUT/'result.json').read_text())
print(json.dumps({k:result[k] for k in ('status','source_commit','diagnostic_denominator','fixed_calls','actual_calls','failures') if k in result},ensure_ascii=False,indent=2))
for name,row in result['videos'].items():
    detection=row.get('detection',{})
    print(name, row['status'], 'best=', detection.get('best'), 'message_unique=', detection.get('top_message_unique'), 'truth_join=',row.get('reporting_only'))
print('Full candidate tables:', OUTPUT/'detections')
